# Ask

## Scenario: 

>"We have customer ratings and review data for over 1,000 products. We're spending a lot on discounts, but we're not sure if discounting is actually making customers happier or driving better ratings. We also don't know which product categories are performing best. Can you look into this for us?"

## Business questions

| # | Business Question | Why It Matters |
| :-- | :--- | :--- |
| 1 | Does a higher discount percentage lead to a higher product rating? | Helps decide if deep discounting is worth the margin loss |
| 2 | Which product categories have the highest average ratings? | Identifies Amazon's strongest and weakest performing verticals |
| 3 | Are heavily discounted products reviewed more frequently? | Determines if discounts drive customer engagement |
| 4 | What is the relationship between actual price and customer rating? | Reveals if price perception affects satisfaction |
| 5 | Which categories receive the most customer reviews (rating count)? | Highlights where customers are most engaged |

# Prepare

## Data Evaluation

- **Reliable**: Low -> Single snapshot, no clear collection methodology stated
- **Original**: Low -> Scraped from Amazon's website
- **Comprehensive**: High -> Has prices, discounts, ratings, reviews, and categories
- **Current**: Low -> Published Jan 2023 (May not reflect today's market)
- **Cited**: High -> Source is publicly documented on Kaggle

## Data Scope

| Column | Type | Relevant? | Note |
| :--- | :--- | :---: | :--- |
| **product_id** | ID | [x] | Unique identifier |
| **product_name** | Text | [x] | For context/labeling |
| **category** | Text | [x] | Key analysis column |
| **discounted_price** | Numeric (text) | [x] | Key analysis column |
| **actual_price** | Numeric (text) | [x] | Key analysis column |
| **discount_percentage** | Numeric (text) | [x] | Key analysis column |
| **rating** | Numeric (text) | [x] | Key analysis column |
| **rating_count** | Numeric (text) | [x] | Key analysis column |
| **about_product** | Long Text | [ ] | Skip for now |
| **user_id** | ID | [ ] | Not needed for our questions |
| **user_name** | Text | [ ] | Not needed |
| **review_id** | ID | [ ] | Not needed |
| **review_title** | Text | [ ] | Skip for now |
| **review_content** | Long Text | [ ] | Skip for now |
| **img_link** | URL | [ ] | Not needed |
| **product_link** | URL | [ ] | Not needed |

## Data Info
- Data source: Kaggle — Amazon Sales Dataset by [KARKAVELRAJA J](https://www.kaggle.com/karkavelrajaj)
- License: CC BY-NC-SA 4.0
- Dataset size: ~1,465 rows × 16 columns
- Key columns for analysis: category, discounted_price, actual_price, 
  discount_percentage, rating, rating_count
- Anticipated issues: currency symbols, % signs, text-type numerics, 
  hierarchical categories
- Limitations: static snapshot (Jan 2023), scraped data (not official)
- Note: Currency symbol in `₹` Indian Rupee (INR)

# Process

## Steps Overview
1. Pre-cleaning inspection  
1.1 Package prep: `import` NumPy, pandas, matplotlib, seaborn, etc.  
1.2 Data loading     
1.3 File management (`pathlib`)  
1.4 Data inspection (first_look func)
2. Data cleaning  
2.1 Clean column names  
2.2 Remove irrelevant columns/scope  
2.3 Remove duplicate  
2.4 Fix data types  
2.5 Standardizing   
2.6 Handle missing values  
2.7 Handle outliers  
2.8 Validate  
2.9 Final QA and save  

In [ ]:
# 1.1 Package prep
import numpy as np  
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings # Keep logs clean

warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# 1.2 Data loading

import kagglehub
import shutil
from pathlib import Path

# --- Download from Kaggle ---
download_path = kagglehub.dataset_download('karkavelrajaj/amazon-sales-dataset')
print(f"Downloaded to cache: {download_path}")

# 1.3 File management

# --- Define project paths ---
base_dir = Path.cwd()
raw_path = base_dir / 'data' / 'raw' / 'amazon.csv'
processed_path = base_dir / 'data' / 'processed' / 'amazon_clean.csv'

# --- Copy raw file into data/raw/ ---
source_file = Path(download_path) / 'amazon.csv'
shutil.copy(source_file, raw_path)
print(f"Raw file saved to: {raw_path}")

# --- Make a copy for cleaning (data/processed/) ---
shutil.copy(raw_path, processed_path)
print(f"Working copy saved to: {processed_path}")

In [ ]:
# 1.4 Data inspection

df = pd.read_csv(processed_path)

def first_look(df):
    print(f"Shape: {df.shape}")
    print(f"\nColumn types:\n{df.dtypes}")
    print(f"\nMissing values (%):\n{(df.isnull().mean() * 100).round(1)}")
    print(f"\nDuplicates: {df.duplicated().sum()}")
    print(f"\nFirst 5 rows:\n{df.head()}")
    print(f"\nBasic stats:\n{df.describe(include='all')}")

first_look(df)

Shape: (1465, 16)

Column types:
product_id             str
product_name           str
category               str
discounted_price       str
actual_price           str
discount_percentage    str
rating                 str
rating_count           str
about_product          str
user_id                str
user_name              str
review_id              str
review_title           str
review_content         str
img_link               str
product_link           str
dtype: object

Missing values (%):
product_id             0.0
product_name           0.0
category               0.0
discounted_price       0.0
actual_price           0.0
discount_percentage    0.0
rating                 0.0
rating_count           0.1
about_product          0.0
user_id                0.0
user_name              0.0
review_id              0.0
review_title           0.0
review_content         0.0
img_link               0.0
product_link           0.0
dtype: float64

Duplicates: 0

First 5 rows:
   product_id          

## Dtype Problems

|Column|Data Types|Expected Dtype|
|:---|:---|:---|
|product_id|`str`||
|product_name|`str`||
|category|`str`||
|discounted_price|`str`|`float`|
|actual_price|`str`|`float`|
|discount_percentage|`str`|`float`|
|rating|`str`|`float`|
|rating_count|`str`|`float`|
|about_product|`str`||
|user_id|`str`|`int`|
|user_name|`str`||
|review_id|`str`|`int`|
|review_title|`str`||
|review_content|`str`||
|img_link|`str`||
|product_link|`str`||

In [8]:
# 2.1 Clean column names

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


In [9]:
# 2.2 Remove irrelevant columns/scope

cols_to_keep = [
  'product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count'
]

df = df[cols_to_keep]

print(f"Shape after dropping irrelevant columns: {df.shape}")

Shape after dropping irrelevant columns: (1465, 8)


In [10]:
# 2.3 Remove duplicate

df = df.drop_duplicates(subset='product_id') # Drop only col that identifies business key
print(f"Shape after dropping duplicate columns: {df.shape}") 

Shape after dropping duplicate columns: (1351, 8)


In [ ]:
# 2.4 Fix data types

df['discounted_price'] = df['discounted_price'].str.replace('₹', '').str.replace(',', '').astype(float)
df['actual_price'] = df['actual_price'].str.replace('₹', '').str.replace(',', '').astype(float)
df['discount_percentage'] = df['discount_percentage'].str.replace('%', '').astype(float)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating_count'] = df['rating_count'].str.replace(',', '').astype(float)

df.dtypes

product_id                 str
product_name               str
category                   str
discounted_price       float64
actual_price           float64
discount_percentage    float64
rating                 float64
rating_count           float64
dtype: object

In [18]:
# 2.5 Standardizing

# Extract top-level category (e.g. "Computers&Accessories|Accessories&Peripherals|..." -> "Computer&Accessories")

df['main_category'] = df['category'].str.split('|').str[0].str.strip()

print(df['main_category'].value_counts())

main_category
Electronics              490
Home&Kitchen             448
Computers&Accessories    375
OfficeProducts            31
MusicalInstruments         2
HomeImprovement            2
Toys&Games                 1
Car&Motorbike              1
Health&PersonalCare        1
Name: count, dtype: int64


In [ ]:
# 2.6 Handle missing values

# Find the missing values
print(f"NULLs before:\n{df.isnull().sum()}")

# Drop rows where key analysis columns are NULL
df = df.dropna(subset=['rating', 'rating_count', 'discount_percentage'])

print(f"\nNULLs after:\n{df.isnull().sum()}")
print(f"Shape: {df.shape}") # Note: If 50% of rows gone -> file corrupt OR too strict subset

NULLs before:
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 1
rating_count           2
main_category          0
dtype: int64

NULLs after:
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           0
main_category          0
dtype: int64
Shape: (1348, 9)


In [ ]:
# 2.7 Handle Outliers

# Find the outlier (Focus on Min/Max)
print(df[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']].describe())

       discounted_price   actual_price  discount_percentage       rating  \
count       1348.000000    1348.000000          1348.000000  1348.000000   
mean        3310.267188    5700.506395            46.662463     4.091988   
std         7180.901370   11229.356113            21.607716     0.295139   
min           39.000000      39.000000             0.000000     2.000000   
25%          349.000000     899.000000            31.000000     3.900000   
50%          899.000000    1795.000000            49.000000     4.100000   
75%         2184.000000    4592.500000            62.000000     4.300000   
max        77990.000000  139900.000000            94.000000     5.000000   

        rating_count  
count    1348.000000  
mean    17656.847923  
std     42158.843602  
min         2.000000  
25%      1107.500000  
50%      4740.000000  
75%     16051.500000  
max    426973.000000  


In [21]:
# Filter based on findings

df = df[df['rating'].between(1, 5)] # If > 5 and < 1 = invalid
df = df[df['discount_percentage'].between(0, 100)] # If over 100 = invalid
df = df[df['rating_count'] > 0] # If 0 = useless for analysis

print(f"Shape after outlier removed: {df.shape}")

Shape after outlier removed: (1348, 9)


Shape before (1348, 9) + Shape after (1348, 9) = No outlier

In [ ]:
# 2.8 Validate

# Sanity checks
assert df['rating'].between(1, 5).all(), "Invalid ratings found!"
assert df['discount_percentage'].between(0, 100).all(), "Invalid discounts found!"
assert df['rating_count'].gt(0).all(), "Zero rating count found!"
assert df.duplicated(subset='product_id').sum() == 0, "Duplicates still exist"

print("🟢 Data is clean!")

✅ Data is clean!


In [24]:
# 2.9 Final QA & Save

print(f"Final Shape: {df.shape}")
print(f"\nFinal Data types:\n{df.dtypes}")
print(f"\nMissing values:]\n{df.isnull().sum()}")

# save
df.to_csv(processed_path, index=False)
print("\n ✅ Cleaned dataset saved to data/processed/amazon_clean.csv")

Final Shape: (1348, 9)

Final Data types:
product_id                 str
product_name               str
category                   str
discounted_price       float64
actual_price           float64
discount_percentage    float64
rating                 float64
rating_count           float64
main_category           object
dtype: object

Missing values:]
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           0
main_category          0
dtype: int64

 ✅ Cleaned dataset saved to data/processed/amazon_clean.csv


# Analyze

## Exploratory Data Analysis